# 04 — Explore the charts

**The question (F4):** what is *the shape of game review scores*? Using **IGDB critic ratings** (switched from Metacritic so recent years aren't starved — IGDB's critic coverage keeps up). 6,005 games with a critic aggregate backed by ≥3 outlets.

**Headline facts (`03-prepare`):** mean 73.5, median 75.3; the 90+ club is only ~246 games (~4%); the 70s are the fat middle (~2,100); sub-50 is rare (~260, ~4%). A **left-skewed pile in the 70s–80s with a thin elite tail and a surprisingly thin bad-score tail**.

Three charts explored below (display-only). **⭐ Review surface — confirm the framing before I build the polished social/web renders in `06-viz-social`.**

*Rendering note: this venv (Py3.14 + matplotlib) hits the known RecursionError on matplotlib ticks, so exploration uses the shared Pillow templates — the real publication charts — rendered inline.*

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
SHARED = PROJECT.parent.parent / 'shared'
sys.path.insert(0, str(SHARED))

import duckdb
from colors import c
from chart_templates import histogram, line_chart, scatter_plot
from IPython.display import display

from src.ingest import load_config
cfg = load_config('config.yaml')
con = duckdb.connect(cfg['settings']['duckdb_file'], read_only=True)

df_all = con.execute('SELECT critic_rating, user_rating, release_year FROM chart_critic_all').df()
print('games:', len(df_all))
df_all['critic_rating'].describe()

## Chart 1 (HERO) — distribution of IGDB critic ratings

Full 0–100 range, 5-point bands, mean + median lines, and the **90+ club highlighted in aqua** (deliberately NOT gold — the gold dashed line is the median, and we don't want the highlight to read as related to it). This is the F4 hero: "the shape of game review scores."

In [ ]:
img = histogram(
    df_all, value_col='critic_rating',
    bin_edges=list(range(0, 101, 5)), x_range=(0, 100), x_tick_step=10,
    x_axis_label='IGDB critic rating', y_axis_label='Number of games',
    bar_color=c('teal'), highlight_range=(90, 100, c('aqua')),
    title='Video game critic review scores',
    subtitle='Distribution of IGDB critic ratings — 6,005 games (≥ 3 critic scores each); each bar = a 5-point band. 90+ highlighted.',
    source='IGDB',
)
display(img)

## Chart 1b (HERO, user side) — distribution of IGDB USER ratings

The same histogram treatment applied to the **community/user** rating instead of the critic aggregate — the natural companion to the hero, and **same colors** (teal bars, aqua 90+ band) so the pair reads as one set; the title carries the critic-vs-user distinction. Same 5-point bands and full 0–100 range, so the two are directly comparable. Users draw from a different pool of games (only those with a user rating), so the count and shape differ from the critic histogram — the interesting question is whether users pile up in the same 70s–80s or spread wider/lower.

In [ ]:
df_user = df_all[df_all['user_rating'].notna()]
print('games with a user rating:', len(df_user))
print(df_user['user_rating'].describe().round(1).to_string())
img = histogram(
    df_user, value_col='user_rating',
    bin_edges=list(range(0, 101, 5)), x_range=(0, 100), x_tick_step=10,
    x_axis_label='IGDB user rating', y_axis_label='Number of games',
    bar_color=c('teal'), highlight_range=(90, 100, c('aqua')),
    title='Video game user review scores',
    subtitle=f'Distribution of IGDB user ratings — {len(df_user):,} games with a community rating; each bar = a 5-point band. 90+ highlighted.',
    source='IGDB',
)
display(img)

## Chart 2 — games scoring 90+ per year

The elite tail over time. Cut at the last complete year (recent releases are still accruing critic scores, so a partial year would look like a false collapse) — the cutoff + reason are stated in the subtitle.

In [ ]:
by_year = con.execute('SELECT year, n_90plus FROM chart_90plus_by_year ORDER BY year').df()
cut = int(by_year['year'].max())
print('spans', int(by_year['year'].min()), '→', cut)
img = line_chart(
    by_year, x_col='year',
    series=[{'col': 'n_90plus', 'label': '90+ games', 'color': c('teal')}],
    x_axis_label='Release year', y_axis_label='Games scoring 90+',
    y_min=0, value_labels=True, label_last=False, markers=True,
    title='Video game review scores — games scoring 90+ by release year',
    subtitle=f'IGDB critic rating. Cut at {cut}: newer releases are still accruing critic scores, so later years would undercount.',
    source='IGDB',
)
display(img)

## Chart 2b — average critic vs average user rating by year (two lines)

One line for the mean **critic** rating and one for the mean **user** rating, per release year — do the two audiences track each other over time, and is one consistently higher? Averages are computed inline from `chart_critic_all` (exploration; if this chart is kept, its series get a proper `chart_` table in `03-prepare` before `06-viz-social`). Restricted to years with enough scored games to be stable, and cut at the same last-complete year as the 90+ line so recent partial years don't distort the tail. User means only cover games that have a user rating.

In [ ]:
avg_by_year = con.execute(f'''
  SELECT release_year AS year,
         ROUND(AVG(critic_rating), 1) AS avg_critic,
         ROUND(AVG(user_rating), 1)   AS avg_user,
         COUNT(*) AS n_games
  FROM chart_critic_all
  WHERE release_year IS NOT NULL
  GROUP BY release_year
  HAVING COUNT(*) >= 10 AND release_year <= {cut}
  ORDER BY release_year
''').df()
print('spans', int(avg_by_year['year'].min()), '→', int(avg_by_year['year'].max()))
print(avg_by_year.to_string(index=False))
img = line_chart(
    avg_by_year, x_col='year',
    series=[
        {'col': 'avg_critic', 'label': 'Critic', 'color': c('teal')},
        {'col': 'avg_user',   'label': 'User',   'color': c('caramel')},
    ],
    x_axis_label='Release year', y_axis_label='Average rating (0–100)',
    y_min=60, y_max=90, markers=True, label_last=True,
    title='Video game review scores — average critic vs user rating by year',
    subtitle=f'IGDB ratings, mean per release year. Years with ≥ 10 scored games, through {cut}.',
    source='IGDB',
)
display(img)

## Chart 3 — user rating vs critic rating (outliers labelled)

Do IGDB users agree with critics? Point per game (the ~5,300 with both a critic and a user rating), with a dashed **y = x** agreement line. The **outliers** — the games that stand most alone on the plot — are coloured a warm **rust** (spice) and labelled with the game name over a white pill so the text reads over the point cloud; everything else stays teal. "Outlier" here = the most **visually isolated** points: for each game we measure the mean distance to its 8 nearest neighbours in normalized score space, and label the 12 with the most empty space around them. That matches what the eye reads as ‘standing out’ — it catches the lonely dots in every corner (sparse upper-left, lower-left, and the odd straggler) and, unlike a distance-from-the-trend measure, does NOT over-pick the tightly-packed cluster in the lower-right where the points have close neighbours.

*One row is dropped: **Infernal** has a critic rating of 0.0 “backed by” 6 outlets — an aggregate of exactly zero over 6 critics is not a real mean, it's an IGDB data gap (it would otherwise be the most isolated dot of all), so it's filtered out of the scatter rather than labelled.*

In [ ]:
df_scatter = con.execute('''
  SELECT name, critic_rating, user_rating
  FROM chart_critic_all
  WHERE user_rating IS NOT NULL
    AND critic_rating > 0   -- drop the single critic=0 artifact (Infernal:
                            -- aggregate 0.0 over 6 outlets is not a real mean;
                            -- it's an IGDB gap, and it reads as a lonely far-left dot)
''').df()
print('games with both ratings (critic>0):', len(df_scatter))
corr = df_scatter['critic_rating'].corr(df_scatter['user_rating'])
print('Pearson r:', round(corr, 3))

# Outliers = the most VISUALLY ISOLATED points — the ones with the most empty
# space around them, which is what the eye actually reads as ‘standing out’.
# Measure: mean distance to each point's K nearest neighbours in normalized
# (0–1 per axis) score space, so both axes weigh equally and 'isolation' means
# the same thing everywhere on the plot. Dense-cloud points have tiny neighbour
# distances; a lonely dot in open space has a large one. This flags the sparse
# corners in EVERY region (upper-left, lower-left, lone stragglers) and does NOT
# over-pick the tightly-packed lower-right cluster, unlike the residual metric.
import numpy as np
N_OUTLIERS = 12
K = 8
cx = df_scatter['critic_rating'].values.astype(float)
uy = df_scatter['user_rating'].values.astype(float)
nx = (cx - cx.min()) / (cx.max() - cx.min())
ny = (uy - uy.min()) / (uy.max() - uy.min())
P = np.column_stack([nx, ny])
n = len(P)
iso = np.empty(n)
for i in range(0, n, 500):                     # chunked brute-force kNN (no scipy)
    blk = P[i:i+500]
    d2 = ((blk[:, None, :] - P[None, :, :]) ** 2).sum(2)
    d2.sort(axis=1)
    iso[i:i+500] = np.sqrt(d2[:, 1:K+1]).mean(1)  # skip self (col 0)
df_scatter['isolation'] = iso
outlier_idx = df_scatter['isolation'].nlargest(N_OUTLIERS).index
df_scatter['is_outlier'] = df_scatter.index.isin(outlier_idx)
print(f'\nlabelled outliers (most isolated — mean dist to {K} nearest neighbours):')
print(df_scatter.loc[outlier_idx, ['name','critic_rating','user_rating','isolation']]
      .reindex(df_scatter.loc[outlier_idx,'isolation'].sort_values(ascending=False).index)
      .round(3).to_string(index=False))

img = scatter_plot(
    df_scatter, x_col='critic_rating', y_col='user_rating',
    x_axis_label='IGDB critic rating', y_axis_label='IGDB user rating',
    point_color=c('teal'),
    label_col='name', outlier_col='is_outlier',
    outlier_color=c('spice'), label_outliers_only=True,
    ref_lines=[{'kind': 'diagonal', 'color': c('gray'), 'label': 'users = critics'}],
    title='Video game review scores — IGDB user rating vs critic rating',
    subtitle=f'{len(df_scatter):,} games with both ratings (0–100). Pearson r = {corr:.2f}. Highlighted = the {N_OUTLIERS} most isolated games (most empty space around them).',
    source='IGDB',
)
display(img)

## Score-band table (caption numbers)

In [ ]:
con.execute('''
SELECT
  CASE
    WHEN critic_rating >= 90 THEN '90-100 (elite)'
    WHEN critic_rating >= 80 THEN '80-89 (great)'
    WHEN critic_rating >= 70 THEN '70-79 (good)'
    WHEN critic_rating >= 60 THEN '60-69 (mixed)'
    WHEN critic_rating >= 50 THEN '50-59 (weak)'
    ELSE 'below 50 (bad)'
  END AS band,
  COUNT(*) AS n_games,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM chart_critic_all
GROUP BY band ORDER BY MIN(critic_rating) DESC
''').df()

---
## Status

**Confirmed for social** (owner, this session):
- **Video game critic review scores** (Chart 1) — histogram, teal bars, aqua 90+.
- **Video game user review scores** (Chart 1b) — histogram, **same colors**, distinction carried in the title.
- **Avg critic vs user by year** (Chart 2b) — the two-line chart (critic + user).
- **User-vs-critic scatter** (Chart 3) — outliers coloured navy + labelled with the game name.

Still open / not part of the social set: the **90+ per year line** (Chart 2) — kept here for reference, not confirmed for publication.

Next: build the finalized social + web renders in `06-viz-social` for the four confirmed charts (and add a `chart_avg_by_year` + scatter/outlier `chart_` table in `03-prepare` so the social renders read from the DB, not inline queries).

---
## Cleanup
Close the (read-only) DuckDB connection so the lock is released.

In [ ]:
con.close()
print('connection closed')